In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [2]:
def load_data(file_path):
    sentences = []
    labels = []
    current_sentence = []
    current_labels = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                parts = line.split()
                if len(parts) >= 2:
                    current_sentence.append(parts[0])
                    current_labels.append(parts[1])
            else:
                if current_sentence:
                    sentences.append(current_sentence)
                    labels.append(current_labels)
                    current_sentence = []
                    current_labels = []
    if current_sentence:
        sentences.append(current_sentence)
        labels.append(current_labels)
    return sentences, labels

In [3]:
train_sentences, train_labels = load_data('train_corrected.txt')
test_sentences, test_labels = load_data('test_corrected.txt')
vocab = {"<UNK>": 0}
all_training_words = [w for s in train_sentences for w in s]

print("Vocabular initiating.")
for word in all_training_words:
    if word not in vocab:
        vocab[word] = len(vocab)

print("Extract tagset.")
tag_set = sorted(list(set([t for s in train_labels for t in s])))
tag2idx = {t: i for i, t in enumerate(tag_set)}
idx2tag = {i: t for t, i in tag2idx.items()}

num_tags = len(tag_set)
num_words = len(vocab)

trans_mat = np.zeros((num_tags, num_tags))
emit_mat = np.zeros((num_tags, num_words))
start_probs = np.zeros(num_tags)


for i in range(len(train_labels)):
    tags = train_labels[i]
    words = train_sentences[i]
    
    start_probs[tag2idx[tags[0]]] += 1
    
    for j in range(len(tags)):
        curr_tag = tag2idx[tags[j]]
        word_idx = vocab.get(words[j], 0)
        emit_mat[curr_tag, word_idx] += 1
        
        if j > 0:
            prev_tag = tag2idx[tags[j-1]]
            trans_mat[prev_tag, curr_tag] += 1

Vocabular initiating.
Extract tagset.


In [4]:
print("Raw data to probabilities.")
epsilon = 1e-10
start_probs = (start_probs + epsilon) / (np.sum(start_probs) + epsilon * num_tags)
trans_mat = (trans_mat + epsilon) / (np.sum(trans_mat, axis=1, keepdims=True) + epsilon * num_tags)
emit_mat = (emit_mat + epsilon) / (np.sum(emit_mat, axis=1, keepdims=True) + epsilon * num_words)

start_log = np.log(start_probs)
trans_log = np.log(trans_mat)
emit_log = np.log(emit_mat)

Raw data to probabilities.


In [5]:
def viterbi(words, vocab, tag2idx, idx2tag, start_log, trans_log, emit_log):
    n = len(words)
    num_tags = len(tag2idx)
    
    dp = np.zeros((num_tags, n))
    backpointer = np.zeros((num_tags, n), dtype=int)
    
    first_word_idx = vocab.get(words[0], 0)
    dp[:, 0] = start_log + emit_log[:, first_word_idx]
    
    for t in range(1, n):
        word_idx = vocab.get(words[t], 0)
        for s in range(num_tags):
            trans_probs = dp[:, t-1] + trans_log[:, s]
            best_prev = np.argmax(trans_probs)
            backpointer[s, t] = best_prev
            dp[s, t] = trans_probs[best_prev] + emit_log[s, word_idx]
            
    best_last_tag = np.argmax(dp[:, n-1])
    best_path = [best_last_tag]
    
    for t in range(n-1, 0, -1):
        best_last_tag = backpointer[best_last_tag, t]
        best_path.insert(0, best_last_tag)
        
    return [idx2tag[i] for i in best_path]

In [6]:
print("Predicting..")
all_preds = []
all_trues = []

for i in range(len(test_sentences)):
    words = test_sentences[i]
    true_tags = test_labels[i]
    pred_tags = viterbi(words, vocab, tag2idx, idx2tag, start_log, trans_log, emit_log)
    
    all_preds.extend(pred_tags)
    all_trues.extend(true_tags)

unique_labels = sorted(list(set(all_trues)))

print(f"Accuracy Score: {accuracy_score(all_trues, all_preds) * 100:.2f}%")
print(classification_report(all_trues, all_preds, digits=4, zero_division=0))

Predicting..
Accuracy Score: 84.88%
              precision    recall  f1-score   support

       B-CRD     0.7293    0.5596    0.6333      1006
       B-DAT     0.8675    0.9226    0.8942       788
       B-EVT     0.5427    0.5934    0.5669       182
       B-FAC     0.3465    0.3763    0.3608        93
       B-GPE     0.7733    0.7711    0.7722      1376
       B-LAN     0.0000    0.0000    0.0000         0
       B-LAW     0.4186    0.5143    0.4615        35
       B-LOC     0.4495    0.4448    0.4471       290
       B-MON     0.7956    0.8549    0.8242       255
       B-NOR     0.7465    0.7143    0.7301       903
       B-ORD     0.2635    0.6241    0.3705       141
       B-ORG     0.6594    0.5644    0.6082       854
       B-PER     0.8387    0.5000    0.6265      1446
       B-PRC     0.4430    0.8698    0.5870       192
       B-PRD     0.5650    0.5058    0.5337       868
       B-QTY     0.3725    0.6310    0.4685       271
       B-REG     0.4783    0.7333    0.5789  